# Project 00 · Groq Model Explorer (Reference solution)

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 2.5–3.5 hours |
| 📋 Prerequisites | Lessons 00.01–00.07; `GROQ_API_KEY` in `.env` |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |
| 📄 Spec | [project/README.md](../README.md) (also in [COURSE_OUTLINE.md](../../../../COURSE_OUTLINE.md#project-00--groq-model-explorer)) |

**Goal:** a notebook that tells you, *today*, which Groq chat models are live and how they compare on the same prompts.

This is the **reference solution**. Try the [starter notebook](../00-project-groq-model-explorer.ipynb) first; compare afterwards.

## Requirements (from the spec)

1. Fetch the live model list from `https://api.groq.com/openai/v1/models`. Show a table of id, owner, context window (when available), and whether the ID appears on the [deprecations](https://console.groq.com/docs/deprecations) page (a small hand-curated list).
2. Pick at least 2 live chat models and send the same 3 prompts (factual, reasoning, creative). Record latency, token usage (`usage_metadata`), and the answer.
3. Save results to `data/outputs/00-model-explorer.csv` (gitignored).
4. Finish with a Markdown **Findings** cell and one entry in `docs/decision-log.md`.

**Acceptance criteria:** Restart & Run All passes with only `GROQ_API_KEY` set · no model ID is hard-coded without a comment saying where and when it was verified · no secrets appear in outputs.

## Setup

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-groq>=1.1.3,<2" "pandas>=3.0.6,<4"

In [ ]:
# Setup 1: Find the repository root.
# WHY: Jupyter starts this kernel in the notebook's own folder (modules/00-foundations/),
# but .env, data/ and src/ live at the repo root, two levels up.
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find the course repo (pyproject.toml). "
        "Start Jupyter from inside your clone of langchain-langgraph-mastery (or set LC_MASTERY_ROOT to it)."
    )

# Setup 2: Import the course package `mastery` from THIS clone's src/.
# WHY: pytest imports it from src/ too (pythonpath = ["src"] in pyproject.toml), so notebooks
# and tests run the same code even if this env was installed from another copy of the repo.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")  # The env's own install; terminal commands use it.
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root :", REPO_ROOT)
print("mastery   :", mastery.__version__, "from", mastery.__file__)

# Setup 3: Check that the env's own install (create-course-env guide, Step 5) points here too.
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder.")
    print("   Kernel → Restart Kernel, then run the notebook from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
import json
import re
import time

import pandas as pd
from IPython.display import Markdown, display
from langchain.chat_models import init_chat_model

from mastery.config import MissingSettingError, load_env, load_settings, require_env
from mastery.groq_catalog import GroqAPIError, fetch_groq_models, looks_like_chat_model, parse_models_payload

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)
try:
    require_env("GROQ_API_KEY", dotenv_path=dotenv_path)
    HAVE_GROQ_KEY = True
    print("✅ GROQ_API_KEY is set. Live mode.")
except MissingSettingError as error:
    HAVE_GROQ_KEY = False
    print("⚠️ " + str(error))
    print("\n➡️ OFFLINE DEMO MODE: the model table uses data/samples/groq-models-sample.json "
          "(an illustrative sample, not live data) and model calls are skipped.")

OUTPUT_CSV = REPO_ROOT / "data" / "outputs" / "00-model-explorer.csv"   # data/outputs/ is gitignored.

---
## Step 1 · Configuration

Everything you might want to change lives in one cell. Model IDs come from `.env`, so a model swap needs no code change.

In [ ]:
# Models to compare. They come from .env (MASTERY_MODEL_FAST / MASTERY_MODEL_REASONING),
# so nothing is hard-coded here.
# VERIFIED 2026-10-11: the defaults openai/gpt-oss-20b and openai/gpt-oss-120b are Groq
# production models (https://console.groq.com/docs/models).
MODELS_TO_COMPARE = [settings.model_fast, settings.model_reasoning]

# Three prompts, one per task type. Short on purpose: the free plan allows ~8K tokens/minute.
PROMPTS = {
    "factual": "In one sentence: what does the LangGraph library provide?",
    "reasoning": "A notebook makes 4 model calls of 1,500 tokens each per minute. "
                 "The limit is 8,000 tokens per minute. Is it under the limit? Answer yes or no, then one sentence why.",
    "creative": "Write a two-line rhyme about reading the docs before copying a model ID.",
}

REPEATS = 3                   # Answers vary between runs: one run per prompt is an anecdote. 1 = quick look.
MAX_OUTPUT_TOKENS = 1_024     # Bounds cost and time; reasoning tokens count toward it.
# On the free plan the token limit (~8K tokens/minute) binds long before the 30 requests/minute
# limit (Lesson 00.06). Worst case per call is the prompt + MAX_OUTPUT_TOKENS, about 1.1K tokens,
# so the loop paces by tokens: it waits until the last minute's tokens + the next worst case fit.
TPM_BUDGET = 6_000            # Stay under the 8K tokens/minute limit, with headroom.

# Programmatic graders: each one turns an answer into pass/fail, so "quality" becomes a number.
GRADERS = {
    "factual": lambda answer: bool(re.search(r"\b(graph|stateful|agent|workflow)s?\b", answer, re.I)),
    "reasoning": lambda answer: answer.strip().lower().lstrip("*# ").startswith("yes"),   # 4 x 1,500 = 6,000 < 8,000
    "creative": lambda answer: len([line for line in answer.strip().splitlines() if line.strip()]) == 2,
}

print("Comparing:", MODELS_TO_COMPARE, f"x {len(PROMPTS)} prompts x {REPEATS} repeats",
      f"= {len(MODELS_TO_COMPARE) * len(PROMPTS) * REPEATS} calls")

**Check the graders first.** A grader you haven't tested is a second thing that can be wrong. Every line below should be ✅.

In [ ]:
# 🧪 Offline: the graders on canned answers, so you can trust them before any call.
CANNED = [
    ("factual", "LangGraph provides a framework for stateful, graph-based agent workflows.", True),
    ("factual", "It is a library.", False),
    ("reasoning", "Yes. 4 x 1,500 = 6,000 tokens, under 8,000.", True),
    ("reasoning", "No, it is over the limit.", False),
    ("creative", "Read the docs before you paste,\nor your model ID goes to waste.", True),
    ("creative", "One long line with no rhyme.", False),
]
for task, answer, expected in CANNED:
    got = GRADERS[task](answer)
    print("✅" if got == expected else "❌", f"{task:<10} expected {expected!s:<5} got {got!s:<5} {answer[:45]!r}")

## Step 2 · A curated deprecation list

The Models API tells you what is live; it does **not** tell you what is about to disappear. That's on the [deprecations page](https://console.groq.com/docs/deprecations), so we keep a small, dated copy (Lesson 00.06).

In [ ]:
# Hand-curated from https://console.groq.com/docs/deprecations. VERIFIED 2026-10-11.
# Shutdown dates apply to free and developer tiers. Update this dict at every research checkpoint.
KNOWN_DEPRECATED: dict[str, str] = {
    "groq/compound": "2026-09-21",
    "groq/compound-mini": "2026-09-21",
    "qwen/qwen3.6-27b": "2026-09-14",
    "llama-3.1-8b-instant": "2026-08-16",
    "llama-3.3-70b-versatile": "2026-08-16",
    "qwen/qwen3-32b": "2026-07-17",
    "meta-llama/llama-4-scout-17b-16e-instruct": "2026-07-17",
    "moonshotai/kimi-k2-instruct-0905": "2026-04-15",
    "moonshotai/kimi-k2-instruct": "2025-10-10",
    "gemma2-9b-it": "2025-10-08",
    "meta-llama/llama-4-maverick-17b-128e-instruct": "2026-03-09",
    "meta-llama/llama-guard-4-12b": "2026-03-05",
    "playai-tts": "2025-12-31",
    "playai-tts-arabic": "2025-12-31",
}
print(len(KNOWN_DEPRECATED), "deprecated IDs curated.")

## Step 3 · Live models (Requirement 1)

In offline demo mode the list comes from `data/samples/groq-models-sample.json`, an **illustrative** file built from Groq's docs. The live API is always the source of truth.

In [ ]:
# 🌐 Calls the Groq Models API (no tokens generated). Falls back to the sample file offline.
SAMPLE_FILE = REPO_ROOT / "data" / "samples" / "groq-models-sample.json"

models, source = [], ""
if HAVE_GROQ_KEY:
    try:
        models, source = fetch_groq_models(), "live Groq Models API"
    except GroqAPIError as error:
        print("⚠️", error)
if not models:
    models = parse_models_payload(json.loads(SAMPLE_FILE.read_text(encoding="utf-8")))
    source = "OFFLINE SAMPLE (illustrative, not live)"
print(f"{len(models)} models from: {source}")

In [ ]:
def models_table(models, deprecated: dict[str, str]) -> pd.DataFrame:
    """Requirement 1: one row per model, with a deprecation flag."""
    rows = [{
        "id": m.id,
        "owned_by": m.owned_by,
        "context_window": m.context_window,
        "max_completion_tokens": m.max_completion_tokens,
        "chat_model": looks_like_chat_model(m.id),
        "on_deprecation_list": m.id in deprecated,
        "shutdown_date": deprecated.get(m.id, ""),
    } for m in models]
    return pd.DataFrame(rows).sort_values(["chat_model", "id"], ascending=[False, True], ignore_index=True)


models_df = models_table(models, KNOWN_DEPRECATED)
display(models_df)

In [ ]:
from datetime import date

live_ids = set(models_df["id"])
tag = "" if source.startswith("live") else "🧪 sample: "
for model_id in MODELS_TO_COMPARE:
    if model_id not in live_ids:
        print(f"{tag}❌ {model_id} is not in the model list ({source}). Pick another live chat model in .env.")
    elif model_id in KNOWN_DEPRECATED:
        shutdown = date.fromisoformat(KNOWN_DEPRECATED[model_id])
        days_left = (shutdown - date.today()).days
        if days_left > 0:
            print(f"{tag}⚠️ {model_id} shuts down on {shutdown} ({days_left} days left): migrate now. ({source})")
        else:   # still listed after its shutdown date: usually an enterprise contract or a stale list
            print(f"{tag}⚠️ {model_id} was due to shut down on {shutdown}, yet it's still listed: check your plan "
                  f"and the deprecations page before relying on it. ({source})")
    else:
        print(f"{tag}✅ {model_id} is listed and not on your deprecation list. ({source})")

**You should now see** one ✅ per configured model. The ⚠️ lines only appear when a model you configured is on your deprecation list, which won't happen with today's list: every shutdown on it has already passed, so those models are gone from the live list (and from the offline sample). The check earns its keep the day Groq announces a **future** shutdown for a model you use: add it to `KNOWN_DEPRECATED` at your next research checkpoint and this cell counts the days for you.

## Step 4 · Same prompts, two models (Requirement 2)

Each call is timed with `time.perf_counter()` (a monotonic clock meant for measuring durations) and its `usage_metadata` is recorded. Each prompt runs `REPEATS` times and each answer is graded. The loop paces calls by **tokens** per minute, the limit that binds first on the free plan, not by requests.

In [ ]:
# Called only when HAVE_GROQ_KEY is set (the run cell below checks it).
def run_prompt(model_id: str, task: str, prompt: str) -> dict:
    """Requirement 2: one call, measured. Errors are recorded, not raised, so one bad model can't stop the run."""
    model = init_chat_model(f"groq:{model_id}", temperature=0, reasoning_effort="low",
                            max_tokens=MAX_OUTPUT_TOKENS, timeout=60, max_retries=2)
    start = time.perf_counter()
    try:
        message = model.invoke(prompt)
    except Exception as error:   # noqa: BLE001 - a project report should record any failure
        return {"model": model_id, "task": task, "latency_s": round(time.perf_counter() - start, 3),
                "error": f"{type(error).__name__}: {str(error)[:150]}"}
    usage = message.usage_metadata or {}
    # Groq also reports its own timings: generation time on the server and time spent queued.
    # queue_time can be None, so keep both optional.
    server = message.response_metadata.get("token_usage") or {}
    return {
        "model": model_id,
        "task": task,
        "latency_s": round(time.perf_counter() - start, 3),
        "server_completion_s": server.get("completion_time"),
        "queue_s": server.get("queue_time"),
        "input_tokens": usage.get("input_tokens"),
        "output_tokens": usage.get("output_tokens"),
        "reasoning_tokens": usage.get("output_token_details", {}).get("reasoning"),
        "finish_reason": message.response_metadata.get("finish_reason"),
        "answer": message.text.strip(),
        "error": "",
    }

In [ ]:
# 💸 Calls Groq: models x prompts x REPEATS = 18 calls by default, roughly 6-15K tokens in total.
# Token pacing makes this take ~2-4 minutes on the free plan; REPEATS = 1 takes under a minute.
WORST_CASE_CALL = 100 + MAX_OUTPUT_TOKENS   # short prompt + the output cap


def wait_for_token_budget(spent: list[tuple[float, int]]) -> None:
    """Sleep until the tokens used in the last 60 s plus one worst-case call fit under TPM_BUDGET."""
    while True:
        now = time.monotonic()
        spent[:] = [(t, n) for t, n in spent if now - t < 60]
        if sum(n for _, n in spent) + WORST_CASE_CALL <= TPM_BUDGET:
            return
        time.sleep(max(0.5, 60 - (now - spent[0][0])))


results: list[dict] = []
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: no GROQ_API_KEY (offline demo mode).")
else:
    spent: list[tuple[float, int]] = []
    for repeat in range(1, REPEATS + 1):
        for model_id in MODELS_TO_COMPARE:
            for task, prompt in PROMPTS.items():
                wait_for_token_budget(spent)
                row = run_prompt(model_id, task, prompt)
                used = (row.get("input_tokens") or 0) + (row.get("output_tokens") or 0)
                spent.append((time.monotonic(), used or WORST_CASE_CALL))
                row["repeat"] = repeat
                row["passed"] = not row.get("error") and GRADERS[task](row.get("answer") or "")
                results.append(row)
                print(f"#{repeat} {model_id:<22} {task:<10} {row.get('latency_s', '-')!s:>7}s  "
                      f"{row.get('error') or ('pass' if row['passed'] else 'fail')}")

results_df = pd.DataFrame(results)
display(results_df)

## Step 5 · Save the results (Requirement 3)

In [ ]:
# Requirement 3: save to data/outputs/ (gitignored). Only real results are saved, never the offline demo.
if results_df.empty:
    print("⏭️ Nothing to save (no live results).")
else:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    results_df.assign(run_at=pd.Timestamp.now(tz="UTC").isoformat()).to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Saved {len(results_df)} rows to {OUTPUT_CSV.relative_to(REPO_ROOT)}")

## Step 6 · Summarize

In [ ]:
# Per-model summary: the numbers your Findings cell should discuss.
if results_df.empty or "output_tokens" not in results_df:
    print("⏭️ No live results to summarize.")
else:
    ok = results_df[results_df["error"].fillna("") == ""].copy()
    # Client latency includes network and queueing; the server's completion_time is generation only.
    ok["e2e_tokens_per_s"] = ok["output_tokens"] / ok["latency_s"].where(ok["latency_s"] > 0)
    server_s = pd.to_numeric(ok["server_completion_s"], errors="coerce")
    ok["server_tokens_per_s"] = ok["output_tokens"] / server_s.where(server_s > 0)
    ok["overhead_s"] = ok["latency_s"] - server_s.fillna(0) - pd.to_numeric(ok["queue_s"], errors="coerce").fillna(0)
    summary = ok.groupby("model").agg(
        calls=("task", "count"),
        p50_latency_s=("latency_s", "median"),
        max_latency_s=("latency_s", "max"),
        p50_overhead_s=("overhead_s", "median"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_reasoning_tokens=("reasoning_tokens", "mean"),
        p50_e2e_tokens_per_s=("e2e_tokens_per_s", "median"),
        p50_server_tokens_per_s=("server_tokens_per_s", "median"),
    ).round(2)
    display(summary)
    # Pass rate per model x task. n is tiny (3 runs): a 2/3 vs 3/3 gap is not evidence yet.
    # Module 01 adds confidence intervals; Module 10 runs this on a real dataset.
    pass_rate = results_df.assign(passed=results_df["passed"].astype(bool)).pivot_table(
        index="model", columns="task", values="passed", aggfunc=lambda x: f"{int(x.sum())}/{len(x)}")
    display(pass_rate)
    for row in ok.drop_duplicates(["model", "task"]).itertuples():
        display(Markdown(f"**{row.model} · {row.task}** ({row.latency_s}s)\n\n> {row.answer[:400]}"))

## Stretch goals

1. Add the paid benchmark (`gpt-6-luna`) or Gemini's free tier behind a flag.
2. Plot latency vs tokens.

In [ ]:
# Stretch goals (off by default). Turn a flag on only if you have the key and package.
RUN_PAID_BENCHMARK = False   # needs: %pip install "langchain-openai>=1.7.0,<2" and OPENAI_API_KEY (paid)
RUN_GEMINI = False           # needs: %pip install "langchain-google-genai>=4.4.1,<5" and GOOGLE_API_KEY (free tier: public data only)
# VERIFIED 2026-10-11: gpt-6-luna (developers.openai.com/api/docs/models/gpt-6-luna),
# gemini-3.8-flash (ai.google.dev/gemini-api/docs/models/gemini-3.8-flash).
EXTRA_MODELS = {"openai:gpt-6-luna": RUN_PAID_BENCHMARK, "google_genai:gemini-3.8-flash": RUN_GEMINI}

for model_string, enabled in EXTRA_MODELS.items():
    if not enabled:
        print(f"⏭️ {model_string}: flag is off.")
        continue
    try:
        extra = init_chat_model(model_string, temperature=0)
        for task, prompt in PROMPTS.items():
            start = time.perf_counter()
            message = extra.invoke(prompt)
            tokens_out = (message.usage_metadata or {}).get("output_tokens")
            print(f"{model_string:<32} {task:<10} {time.perf_counter() - start:.2f}s  {tokens_out} out")
    except Exception as error:   # noqa: BLE001 - missing package or key: explain, don't crash
        print(f"⚠️ {model_string}: {type(error).__name__}: {str(error)[:200]}")

In [ ]:
# Stretch: latency vs output tokens. matplotlib is optional (%pip install matplotlib).
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("⏭️ matplotlib not installed; skipping the plot.")

if plt is not None and not results_df.empty and "output_tokens" in results_df:
    fig, ax = plt.subplots(figsize=(6, 4))
    for model_id, group in results_df.dropna(subset=["output_tokens"]).groupby("model"):
        ax.scatter(group["output_tokens"], group["latency_s"], label=model_id)
    ax.set_xlabel("output tokens")
    ax.set_ylabel("latency (s)")
    ax.legend()
    ax.set_title("Groq: latency vs output tokens")
    plt.show()

> ⚠️ **Production note: a one-off comparison is not an evaluation.**
> Three prompts and three runs per model show *speed* and *cost* reliably, and the graders turn quality into a pass rate, but three prompts are not a dataset and 3 runs are too few to separate two models. In production: compare models on a versioned dataset of real tasks with automatic evaluators, repeated runs, and tracked results (LangSmith experiments, Module 10).

> ⚠️ **Production note: the offline sample is not data.**
> `data/samples/groq-models-sample.json` exists so the notebook runs without a key. Never make availability decisions from it: query the live Models API.

## Step 7 · Findings (Requirement 4)

Edit this cell (double-click it). Replace each line with what **your** run showed. Keep it to 5–8 bullets.

- **Live chat models today:** …
- **Deprecated IDs I would have hit by copying the docs:** …
- **Fastest model (median and worst latency) and by how much:** …
- **How much of the latency was network and queueing vs generation (server tokens/s vs end-to-end):** …
- **Token usage, and how much of it was reasoning:** …
- **Pass rate per task (factual / reasoning / creative) and how much it varied across runs:** …
- **My default for Module 01, and why:** …
- **What I would test with more prompts:** …

Draft your decision-log entry, then paste it into [`docs/decision-log.md`](../../../../docs/decision-log.md):

In [ ]:
from datetime import date

best_line = "<fill in after reading the summary>"
if not results_df.empty and "latency_s" in results_df:
    # Only successful calls count: a retired model fails fast, and that is not speed.
    worked = results_df[results_df.get("error", pd.Series("", index=results_df.index)).fillna("") == ""]
    for model_id in MODELS_TO_COMPARE:
        n_calls = int((results_df["model"] == model_id).sum())
        if n_calls and not (worked["model"] == model_id).any():
            print(f"{model_id}: {n_calls}/{n_calls} calls failed -> not a candidate")
    if not worked.empty:
        fastest = worked.groupby("model")["latency_s"].median().idxmin()
        best_line = f"{fastest} had the lowest median latency among models that answered."

print(f"""### D-102 · Default Groq chat model after Project 00 · {date.today().isoformat()}
- Context:    Comparing {", ".join(MODELS_TO_COMPARE)} on 3 prompts (factual, reasoning, creative) x {REPEATS} runs, free plan.
- Options:    {" · ".join(MODELS_TO_COMPARE)}
- Decision:   <your choice>
- Evidence:   data/outputs/00-model-explorer.csv; {best_line}
- Trade-offs: <quality vs latency vs tokens>
- Revisit:    When Groq deprecates one of these models (check console.groq.com/docs/deprecations).
""")

## ✅ Acceptance checklist

- [ ] **Kernel → Restart Kernel and Run All Cells…** passes with only `GROQ_API_KEY` set.
- [ ] `data/outputs/00-model-explorer.csv` exists and has 6 rows (2 models × 3 prompts).
- [ ] Every model ID in the notebook comes from `.env` or carries a `VERIFIED` comment.
- [ ] No key appears in any output (search the notebook for `gsk_`).
- [ ] Findings cell written; decision-log entry pasted.
- [ ] Outputs cleared before committing (`python -m pytest tests/unit/test_notebook_hygiene.py` checks it).

**Next:** [Module 01 · Models, Messages & Prompts](../../../01-models-messages-prompts/). Start with its **Before you start this module** block (pull, install, verify).